In [1]:
import torch

from sonar.inference_pipelines.text import TextToEmbeddingModelPipeline
from sonar.inference_pipelines.text import EmbeddingToTextModelPipeline

sonar_t2v = TextToEmbeddingModelPipeline(
    encoder="text_sonar_basic_encoder", 
    tokenizer="text_sonar_basic_encoder",
    device='cuda'
)
        
sonar_v2t = EmbeddingToTextModelPipeline(
    decoder="text_sonar_basic_decoder", 
    tokenizer="text_sonar_basic_encoder",
    device='cuda'
)

def get_sonar_embedding(text):
    with torch.no_grad():
        embedding = sonar_t2v.predict([text], source_lang="eng_Latn")
    return embedding

### Sonar VS LISA tokens 

#### ReasonSeg data

In [3]:
import os
import json

json_dir = "/home/jovyan/shares/SR006.nfs2/zinkovich/zinkovich/ref-seg-text-break/dataset/reason_seg/ReasonSeg/train"

texts = []

for filename in os.listdir(json_dir):
    if filename.endswith(".json"):
        file_path = os.path.join(json_dir, filename)
        with open(file_path, 'r') as f:
            data = json.load(f)
            if "text" in data:
                texts.extend(data["text"])

print("Extracted texts:")
print(f"Total number of texts: {len(texts)}")


Extracted texts:
Total number of texts: 1326


#### LISA tokens

In [63]:
from transformers import AutoTokenizer

lisa_tokenizer = AutoTokenizer.from_pretrained(
    "xinlai/LISA-13B-llama2-v1",
    use_fast=False,
    add_eos_token=True
)
lisa_tokenizer.pad_token = lisa_tokenizer.unk_token

/home/jovyan/soshin/envs/zinkovich-sonar/lib/python3.11/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [64]:
def lisa_tokenize(text):
    return lisa_tokenizer.encode(text, add_special_tokens=False)

def lisa_decode(tokens):
    return lisa_tokenizer.decode(tokens, skip_special_tokens=True)

text = "Hello, world!"
lisa_tokenize(text)

[15043, 29892, 3186, 29991]

#### SONAR tokens

In [2]:
def sonar_tokenize(text):
    return sonar_t2v.tokenizer.create_encoder(lang="eng_Latn")(text)[1:-1].tolist()

def sonar_decode(tokens):
    return str(sonar_t2v.tokenizer.create_decoder()(torch.tensor(tokens)))

text = "Hello, world!"
sonar_tokenize(text)

[94124, 248079, 15697, 248203]

In [8]:
sonar_t2v.tokenizer.vocab_info.size

256206

In [59]:
from tqdm import tqdm

sonar_vocab = {}
for token in tqdm(range(sonar_t2v.tokenizer.vocab_info.size)):
    subword = sonar_decode([token])
    sonar_vocab[token] = {
        'subword': subword,
        'is_part' : sonar_tokenize(subword) != [token]
    }

100%|██████████| 256206/256206 [00:07<00:00, 34674.58it/s]


##### sonar to lisa transition

In [141]:
sonar2lisa = {}

lisa_vocab = lisa_tokenizer.get_vocab()

special_tokens = [0, 1, 2, 3, 248059]
ignore_tokens = [248059] + list(range(256001, 256206))

sonar2lisa[0] = lisa_tokenizer.pad_token_id
sonar2lisa[1] = lisa_tokenizer.unk_token_id
sonar2lisa[2] = lisa_tokenizer.bos_token_id
sonar2lisa[3] = lisa_tokenizer.eos_token_id

for token, info in tqdm(sonar_vocab.items()):
    if token in special_tokens or token in ignore_tokens: 
        continue

    subword = info['subword']
    if not info['is_part']:
        subword = f'▁{subword}'
        
    if subword in lisa_vocab:
        sonar2lisa[token] = lisa_vocab[subword]
    else:
        sonar2lisa[token] = lisa_tokenize(subword)[0]

  1%|▏         | 3642/256206 [00:00<00:06, 36417.78it/s]

100%|██████████| 256206/256206 [00:13<00:00, 18671.62it/s]


In [151]:
sonar2lisa

{0: 0,
 1: 0,
 2: 1,
 3: 2,
 4: 273,
 5: 302,
 6: 286,
 7: 260,
 8: 413,
 9: 263,
 10: 269,
 11: 264,
 12: 262,
 13: 270,
 14: 261,
 15: 433,
 16: 289,
 17: 282,
 18: 265,
 19: 279,
 20: 271,
 21: 275,
 22: 280,
 23: 321,
 24: 348,
 25: 492,
 26: 288,
 27: 325,
 28: 298,
 29: 274,
 30: 474,
 31: 557,
 32: 294,
 33: 267,
 34: 314,
 35: 343,
 36: 277,
 37: 272,
 38: 330,
 39: 300,
 40: 281,
 41: 284,
 42: 285,
 43: 1299,
 44: 1177,
 45: 529,
 46: 3352,
 47: 8254,
 48: 16173,
 49: 14573,
 50: 341,
 51: 364,
 52: 381,
 53: 574,
 54: 2034,
 55: 301,
 56: 328,
 57: 359,
 58: 290,
 59: 332,
 60: 295,
 61: 432,
 62: 1055,
 63: 375,
 64: 331,
 65: 305,
 66: 638,
 67: 398,
 68: 3249,
 69: 376,
 70: 319,
 71: 941,
 72: 341,
 73: 309,
 74: 259,
 75: 326,
 76: 287,
 77: 1416,
 78: 318,
 79: 316,
 80: 2766,
 81: 869,
 82: 425,
 83: 317,
 84: 405,
 85: 333,
 86: 371,
 87: 292,
 88: 388,
 89: 448,
 90: 29871,
 91: 554,
 92: 336,
 93: 324,
 94: 259,
 95: 259,
 96: 454,
 97: 423,
 98: 476,
 99: 13560,
 

In [158]:
lisa_tokenize(sonar_vocab[979]['subword'])

[29871, 30159, 30162]

In [161]:
lisa_decode([30162])

'ن'

In [153]:
sonar_vocab[979]

{'subword': 'من', 'is_part': False}

In [154]:
sonar_vocab[987]

{'subword': 'સ', 'is_part': False}

In [157]:
[k for k, v in lisa_vocab.items() if v == 259]

['▁▁']

In [150]:
len(set(list(sonar2lisa.values())))

21116

#### Comparison

##### overall lisa vocab
Не очень репрезентативно, так как лезут всякие термины, специфичные для словаря LISA (например, "------" или какие-то названия функций из кода)

In [6]:
import numpy as np

n_special_tokens = np.arange(0, 260)
n_lisa_tokens = 0

matches = {}
mismatches = {}

for lisa_subword, i in lisa_tokenizer.get_vocab().items():
    if i in n_special_tokens:
        continue
    else:
        sonar_tokens = sonar_tokenize(lisa_subword)
        if len(sonar_tokens) == 0:
            continue
        if len(sonar_tokens) == 1:
            matches[lisa_subword] = {
                'sonar_token': sonar_tokens[0],
                'lisa_token' : i
            }
        else:
            mismatches[lisa_subword] = {
                'sonar_tokens': sonar_tokens,
                'lisa_token' : i
            }
        n_lisa_tokens += 1

In [7]:
print(f'matches: {len(matches)} out of {n_lisa_tokens}, p = {len(matches)/n_lisa_tokens * 100:.0f}%')

matches: 19443 out of 31707, p = 61%


In [8]:
s = {}
for key, value in mismatches.items():
    length = len(value['sonar_tokens'])
    s[length] = s.get(length, 0) + 1

In [9]:
sorted(s.items(), key=lambda x: x[0])

[(2, 10829), (3, 1230), (4, 158), (5, 36), (6, 6), (7, 1), (8, 3), (16, 1)]

##### only reasonseg data

In [62]:
sonar_reasonseg_tokens = {}

for text in texts:
    for word in text.split():
        sonar_tokens = sonar_tokenize(word)
        for i, token in enumerate(sonar_tokens):
            subword = sonar_decode([token])
            if i == 0:
                sonar_reasonseg_tokens[subword] = token
            else:
                if '>' in subword:
                    raise ValueError(f"subword {subword} contains '>'")
                sonar_reasonseg_tokens['>' + subword] = token

In [63]:
N = sonar_t2v.tokenizer.vocab_info.size
print(f'tokens in ReasonSeg: {len(sonar_reasonseg_tokens)} out of {N}, p = {len(sonar_reasonseg_tokens)/N * 100:.0f}%')

tokens in ReasonSeg: 3729 out of 256206, p = 1%


In [10]:
lisa_t2s = {t : s for s, t in lisa_tokenizer.get_vocab().items()}
lisa_reasonseg_tokens = {}

for text in texts:
    for word in text.split():
        lisa_tokens = lisa_tokenize(word)
        for token in lisa_tokens:
            lisa_reasonseg_tokens[lisa_t2s[token]] = token

In [11]:
print(f'tokens in ReasonSeg: {len(lisa_reasonseg_tokens)} out of {len(lisa_tokenizer.get_vocab())}, p = {len(lisa_reasonseg_tokens)/len(lisa_tokenizer.get_vocab()) * 100:.0f}%')

tokens in ReasonSeg: 3795 out of 32003, p = 12%


In [72]:
import numpy as np

matches = {}
mismatches = {}

for subword, i in sonar_reasonseg_tokens.items():
    tokens = lisa_tokenize(subword)

    if '>' in subword:
        tokens = tokens[1:]

    if len(tokens) == 1:
        matches[subword] = {
            'lisa_token': tokens[0],
            'sonar_token' : i
        }
    else:
        mismatches[subword] = {
            'lisa_token': tokens,
            'sonar_token' : i
        }

In [73]:
print(f'matches: {len(matches)} out of {len(lisa_reasonseg_tokens)}, p = {len(matches)/len(lisa_reasonseg_tokens) * 100:.0f}%')

matches: 3066 out of 3795, p = 81%


In [74]:
mismatches

{'meetings': {'lisa_token': [5870, 886], 'sonar_token': 202050},
 'portr': {'lisa_token': [2011, 29878], 'sonar_token': 152416},
 'obje': {'lisa_token': [704, 1324], 'sonar_token': 21828},
 '>cts': {'lisa_token': [312, 29879], 'sonar_token': 19592},
 'soldi': {'lisa_token': [5239, 29875], 'sonar_token': 57388},
 'obsta': {'lisa_token': [704, 5173], 'sonar_token': 50415},
 'enta': {'lisa_token': [875, 29874], 'sonar_token': 126415},
 'regula': {'lisa_token': [1072, 2497], 'sonar_token': 28024},
 'crucial': {'lisa_token': [7618, 1455], 'sonar_token': 182071},
 'Suc': {'lisa_token': [317, 1682], 'sonar_token': 60708},
 'survive': {'lisa_token': [10503, 573], 'sonar_token': 215870},
 'illum': {'lisa_token': [4486, 398], 'sonar_token': 146664},
 '>fici': {'lisa_token': [29888, 1654], 'sonar_token': 6570},
 'bri': {'lisa_token': [289, 374], 'sonar_token': 30540},
 'distin': {'lisa_token': [1320, 262], 'sonar_token': 99191},
 '>ctive': {'lisa_token': [312, 573], 'sonar_token': 14825},
 'indic

In [77]:
s = {}
for key, value in mismatches.items():
    length = len(value['lisa_token'])
    s[length] = s.get(length, 0) + 1

In [78]:
sorted(s.items(), key=lambda x: x[0])

[(0, 1), (2, 646), (3, 13), (4, 2), (5, 1)]

In [3]:
chunks = pd.read_csv("/home/jovyan/zinkovich/ref-seg-text-break/experiments/newest-gumbel-exp/nemotron_scores.csv", chunksize=1000)
df = pd.concat(chunks, ignore_index=True)

In [ ]:
import csv
import pandas as pd

file_path = "/home/jovyan/zinkovich/ref-seg-text-break/experiments/newest-gumbel-exp/nemotron_scores.csv"

data = []
with open(file_path, mode='r', newline='', encoding='utf-8') as file:
    csv_reader = csv.reader(file)
    header = next(csv_reader)
    for row in csv_reader:
        data.append(row)
    df = pd.DataFrame(data, columns=header)

In [12]:
csv_reader

In [1]:
text = "During a brainstorming event, it is common to record ideas on a whiteboard."
max_word_length = 15
max_sublength = 8
stop_words = ['.', ',', '?', '!', ' ']

if len(text.split()) > max_word_length:
    for i, letter in enumerate(text):
        if letter in stop_words and \
            len(text[:i].split()) > max_sublength and \
            i + 2 < len(text):

            end_adv_str = i + 1
            break

In [2]:
text[:end_adv_str]

NameError: name 'end_adv_str' is not defined